In [1]:
#!pip install langgraph==0.2.35 langchain==0.2.15 langchain-community==0.2.10 langchain-openai==0.1.23 sentence-transformers==2.2.2 chromadb==0.4.22 faiss-cpu==1.7.4 datasets==2.14.6 huggingface-hub==0.16.4 transformers==4.30.2 accelerate==0.20.3 


In [2]:
from datasets import load_dataset
import json, pathlib

# تحميل البيانات
dt = load_dataset("openai/openai_humaneval")
test = dt["test"]

# حفظها في ملف JSONL داخل مجلد data
out_dir = pathlib.Path("data")
out_dir.mkdir(exist_ok=True)
path = out_dir / "humaneval_test.jsonl"

with open(path, "w", encoding="utf-8") as f:
    for rec in test:
        obj = {
            "task_id": rec["task_id"],
            "prompt": rec["prompt"],
            "canonical_solution": rec["canonical_solution"],
            "entry_point": rec["entry_point"],
        }
        f.write(json.dumps(obj, ensure_ascii=False) + "\n")

print(f"✅ Saved {len(test)} tasks to {path}")


d:\Anaconda\envs\tf-gpu-fix\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Saved 164 tasks to data\humaneval_test.jsonl


In [3]:
import re, json, pathlib

# مسار الملف اللي حفظتيه في الخطوة السابقة
data_path = pathlib.Path("data/humaneval_test.jsonl")

# قراءة البيانات
rows = [json.loads(l) for l in data_path.read_text(encoding="utf-8").splitlines()]
print(f"Loaded {len(rows)} records ✅")

# 🧹 خطوة 1: تنظيف النصوص
def clean_text(text: str) -> str:
    text = re.sub(r"\s+", " ", text)  # إزالة المسافات الزائدة
    text = re.sub(r"```python|```", "", text)  # إزالة علامات الكود
    text = text.strip()
    return text

for r in rows:
    r["prompt"] = clean_text(r["prompt"])
    r["canonical_solution"] = clean_text(r["canonical_solution"])

# ✂️ خطوة 2: تقسيم النصوص إلى Chunks
def chunk_text(text, max_length=150, overlap=30):
    """تقسيم النص إلى مقاطع متداخلة (chunks)"""
    words = text.split()
    chunks = []
    start = 0
    while start < len(words):
        end = min(start + max_length, len(words))
        chunk = " ".join(words[start:end])
        chunks.append(chunk)
        start += max_length - overlap
    return chunks

# إنشاء قائمة من المقاطع لكل مهمة
processed_docs = []
for r in rows:
    content = f"Task ID: {r['task_id']}\nPrompt: {r['prompt']}\nSolution: {r['canonical_solution']}"
    for chunk in chunk_text(content):
        processed_docs.append({
            "task_id": r["task_id"],
            "entry_point": r["entry_point"],
            "chunk": chunk
        })

print(f"✅ Total Chunks created: {len(processed_docs)}")
print("Example chunk:\n", processed_docs[0]["chunk"][:500])

# حفظ النتائج في ملف جديد
path_chunks = pathlib.Path("data/humaneval_chunks.jsonl")
with open(path_chunks, "w", encoding="utf-8") as f:
    for rec in processed_docs:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

print(f"✅ Saved preprocessed & chunked data to {path_chunks}")


Loaded 164 records ✅
✅ Total Chunks created: 208
Example chunk:
 Task ID: HumanEval/0 Prompt: from typing import List def has_close_elements(numbers: List[float], threshold: float) -> bool: """ Check if in given list of numbers, are any two numbers closer to each other than given threshold. >>> has_close_elements([1.0, 2.0, 3.0], 0.5) False >>> has_close_elements([1.0, 2.8, 3.0, 4.0, 5.0, 2.0], 0.3) True """ Solution: for idx, elem in enumerate(numbers): for idx2, elem2 in enumerate(numbers): if idx != idx2: distance = abs(elem - elem2) if distance < threshol
✅ Saved preprocessed & chunked data to data\humaneval_chunks.jsonl


In [4]:
from langchain_community.vectorstores import Chroma
from langchain.docstore.document import Document
from langchain_community.embeddings import HuggingFaceEmbeddings
import json, pathlib

# إعداد النموذج ومجلد التخزين
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
DB_DIR = "chroma_store"

# تحميل البيانات بعد الـ chunking
data_path = pathlib.Path("data/humaneval_chunks.jsonl")
rows = [json.loads(l) for l in data_path.read_text(encoding="utf-8").splitlines()]
print(f"📄 Loaded {len(rows)} chunked records")

# إنشاء مستندات للـ Vector Store
docs = []
for r in rows:
    text = r["chunk"]
    meta = {"task_id": r["task_id"], "entry_point": r["entry_point"]}
    docs.append(Document(page_content=text, metadata=meta))

# إنشاء embeddings باستخدام sentence-transformers
emb = HuggingFaceEmbeddings(model_name=MODEL_NAME)

# إنشاء قاعدة البيانات Chroma
vectordb = Chroma.from_documents(documents=docs, embedding=emb, persist_directory=DB_DIR)
vectordb.persist()

print(f"✅ RAG Store built with {len(docs)} chunks at '{DB_DIR}'")


📄 Loaded 208 chunked records


C:\Users\ayaha\AppData\Local\Temp\ipykernel_10680\631417942.py:23: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 0.3.0. An updated version of the class exists in the langchain-huggingface package and should be used instead. To use it run `pip install -U langchain-huggingface` and import as `from langchain_huggingface import HuggingFaceEmbeddings`.
  emb = HuggingFaceEmbeddings(model_name=MODEL_NAME)
d:\Anaconda\envs\tf-gpu-fix\lib\site-packages\accelerate\utils\torch_xla.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() take

✅ RAG Store built with 208 chunks at 'chroma_store'


C:\Users\ayaha\AppData\Local\Temp\ipykernel_10680\631417942.py:27: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  vectordb.persist()


In [ ]:
from typing import TypedDict, List, Dict, Any
from langchain_openai import ChatOpenAI
from langchain.schema import SystemMessage, HumanMessage
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langgraph.graph import StateGraph, END

# 🔑 تأكدي أنك ضيفه الـ OPENAI_API_KEY في بيئة العمل
import os

# نموذج المحادثة
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.1)

# الحالة (State)
class AssistantState(TypedDict, total=False):
    user_query: str
    intent: str
    retrieved: List[Dict[str, Any]]
    reformulations: int
    draft: str
    final_answer: str


In [6]:
DB_DIR = "chroma_store"
EMB_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

def classify_intent(state: Dict[str, Any]) -> Dict[str, Any]:
    q = state["user_query"]
    sys = SystemMessage(content="Classify the user's intent as 'generate' or 'explain'. Return one word only.")
    res = llm.invoke([sys, HumanMessage(content=q)])
    text = res.content.lower()
    state["intent"] = "generate" if "generate" in text else "explain"
    return state

def retrieve_examples(state: Dict[str, Any]) -> Dict[str, Any]:
    emb = HuggingFaceEmbeddings(model_name=EMB_MODEL)
    vectordb = Chroma(persist_directory=DB_DIR, embedding_function=emb)
    hits = vectordb.similarity_search(state["user_query"], k=4)
    state["retrieved"] = [{"text": h.page_content, "meta": h.metadata} for h in hits]
    return state

def evaluate_retrieval(state: Dict[str, Any]) -> Dict[str, Any]:
    state["needs_reformulate"] = not bool(state.get("retrieved"))
    return state

def reformulate_query(state: Dict[str, Any]) -> Dict[str, Any]:
    res = llm([SystemMessage(content="Rephrase the query for better retrieval."), HumanMessage(content=state["user_query"])])
    state["user_query"] = res.content.strip()
    state["reformulations"] = state.get("reformulations", 0) + 1
    return state

def reasoning(state: Dict[str, Any]) -> Dict[str, Any]:
    """
    توليد الكود فقط (بدون شرح) بناءً على نية المستخدم.
    """
    intent = state["intent"]
    retrieved_txt = "\n\n---\n".join([r["text"] for r in state.get("retrieved", [])])
    
    # لو النية توليد كود
    if intent == "generate":
        sys_msg = "You are a helpful Python coding assistant. Respond with Python code only — no explanation or markdown."
        prompt = f"User wants to generate code:\n{state['user_query']}\n\nUse these references if relevant:\n{retrieved_txt}"
    else:
        sys_msg = "You are a Python tutor. Explain clearly but respond mainly with code examples."
        prompt = f"Explain the following:\n{state['user_query']}\n\nReferences:\n{retrieved_txt}"
    
    res = llm.invoke([SystemMessage(content=sys_msg), HumanMessage(content=prompt)])
    state["draft"] = res.content.strip()
    return state


def finalize(state: Dict[str, Any]) -> Dict[str, Any]:
    text = state.get("draft", "")
    clean_code = text.replace("```python", "").replace("```", "").strip()
    import re
    clean_code = re.sub(r'""".*?"""', '', clean_code, flags=re.DOTALL).strip()
    state["final_answer"] = clean_code
    return state



In [7]:
def build_graph():
    g = StateGraph(AssistantState)
    g.add_node("classify", classify_intent)
    g.add_node("retrieve", retrieve_examples)
    g.add_node("evaluate", evaluate_retrieval)
    g.add_node("reformulate", reformulate_query)
    g.add_node("reasoning", reasoning)
    g.add_node("finalize", finalize)

    g.set_entry_point("classify")
    g.add_edge("classify", "retrieve")
    g.add_edge("retrieve", "evaluate")

    def route_after_eval(state):
        return "reformulate" if state.get("needs_reformulate") and state.get("reformulations", 0) < 2 else "reasoning"

    g.add_conditional_edges("evaluate", route_after_eval,
                            {"reformulate": "reformulate", "reasoning": "reasoning"})
    g.add_edge("reformulate", "retrieve")
    g.add_edge("reasoning", "finalize")
    g.add_edge("finalize", END)
    return g.compile()

app = build_graph()


In [8]:
query = "Write Python code to find the longest palindromic substring in a given string."
result = app.invoke({"user_query": query})


print("✅ Final Answer:\n")
print(result["final_answer"])

C:\Users\ayaha\AppData\Local\Temp\ipykernel_10680\3788542011.py:14: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 0.4. An updated version of the class exists in the langchain-chroma package and should be used instead. To use it run `pip install -U langchain-chroma` and import as `from langchain_chroma import Chroma`.
  vectordb = Chroma(persist_directory=DB_DIR, embedding_function=emb)
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


✅ Final Answer:

from typing import Optional

def longest_palindrome(s: str) -> Optional[str]:
    def is_palindrome(sub: str) -> bool:
        return sub == sub[::-1]

    n = len(s)
    if n == 0:
        return None

    longest = ""
    for i in range(n):
        for j in range(i, n):
            substring = s[i:j + 1]
            if is_palindrome(substring) and len(substring) > len(longest):
                longest = substring

    return longest if longest else None


In [9]:
query = "write Python code to check if a number is prime."
result = app.invoke({"user_query": query})
print("✅ Final Answer:\n")
print(result["final_answer"])


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


✅ Final Answer:

def is_prime(n):
    
    if n < 2:
        return False
    for k in range(2, int(n**0.5) + 1):
        if n % k == 0:
            return False
    return True


In [10]:
query = "اشرح لي how to print hello world"
result = app.invoke({"user_query": query})
print("✅ Final Answer:\n")
print(result["final_answer"])


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


✅ Final Answer:

To print "Hello, World!" in Python, you can use the `print()` function. Here’s a simple example:


print("Hello, World!")


When you run this code, it will output:


Hello, World!


If you have any specific requirements or additional context regarding the references you provided, please let me know!


In [11]:
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

emb = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectordb = Chroma(persist_directory=r"D:\NLP\codes\TASK 5\chroma_store", embedding_function=emb)

queries = [
    "python code to check if number is prime",
    "reverse a string in python",
    "calculate factorial using recursion"
]

for q in queries:
    results = vectordb.similarity_search(q, k=3)
    print(f"\n🔍 Query: {q}")
    for r in results:
        print("-", r.metadata["task_id"], "→", r.page_content[:80].replace("\n", " "), "...")


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given



🔍 Query: python code to check if number is prime
- HumanEval/31 → Task ID: HumanEval/31 Prompt: def is_prime(n): """Return true if a given number  ...
- HumanEval/31 → Task ID: HumanEval/31 Prompt: def is_prime(n): """Return true if a given number  ...
- HumanEval/31 → Task ID: HumanEval/31 Prompt: def is_prime(n): """Return true if a given number  ...

🔍 Query: reverse a string in python
- HumanEval/161 → Task ID: HumanEval/161 Prompt: def solve(s): """You are given a string s. if s[i ...
- HumanEval/161 → Task ID: HumanEval/161 Prompt: def solve(s): """You are given a string s. if s[i ...
- HumanEval/161 → Task ID: HumanEval/161 Prompt: def solve(s): """You are given a string s. if s[i ...

🔍 Query: calculate factorial using recursion
- HumanEval/107 → odd_palindrome_count = 0 for i in range(1, n+1): if i%2 == 1 and is_palindrome(i ...
- HumanEval/107 → odd_palindrome_count = 0 for i in range(1, n+1): if i%2 == 1 and is_palindrome(i ...
- HumanEval/107 → odd_palindrome_count = 0 for

In [12]:
import re, traceback, random

def auto_generate_tests(func_name):
    """ينشئ test cases مناسبة حسب نوع الدالة"""
    func_name = func_name.lower()
    
    # 🔹 Sorting functions
    if "sort" in func_name:
        return [
            (([5, 2, 3, 1],), [1, 2, 3, 5]),
            (([10, -1, 3],), [-1, 3, 10]),
            (([],), []),
            (([1],), [1]),
        ]
    
    # 🔹 Prime-number functions
    elif "prime" in func_name:
        return [
            ((2,), True),
            ((4,), False),
            ((11,), True),
            ((15,), False),
            ((101,), True),
        ]
    
    # 🔹 Reverse-string functions
    elif "reverse" in func_name:
        return [
            (("hello",), "olleh"),
            (("Aya",), "ayA"[::-1]),  # just "ayA" reversed = "Aya" backwards
            (("",), ""),
        ]
    
    # 🔹 Factorial functions
    elif "factorial" in func_name:
        return [
            ((0,), 1),
            ((1,), 1),
            ((5,), 120),
            ((6,), 720),
        ]
    
    # 🔹 Palindrome functions
    elif "palindrome" in func_name:
        return [
            (("racecar",), True),
            (("hello",), False),
            (("madam",), True),
        ]
    
    # 🔹 Fibonacci functions
    elif "fibo" in func_name:
        return [
            ((1,), 1),
            ((5,), 5),
            ((7,), 13),
        ]
    
    # 🔹 Default numeric test
    else:
        return [((random.randint(1, 10),), None)]


def auto_evaluate_generated_code(code: str):
    """يقيم الكود تلقائيًا ويطبع النتائج"""
    namespace = {}
    clean_code = code.strip().replace("```python", "").replace("```", "")
    
    try:
        exec(clean_code, namespace)
    except Exception as e:
        print("❌ Error executing generated code:")
        traceback.print_exc()
        return 0
    
    # اكتشاف الدوال
    func_names = [name for name, obj in namespace.items() if callable(obj)]
    if not func_names:
        print("⚠️ No functions found.")
        return 0
    
    best_acc = 0
    for func_name in func_names:
        func = namespace[func_name]
        test_cases = auto_generate_tests(func_name)
        
        correct = 0
        for inp, expected in test_cases:
            try:
                output = func(*inp)
                # لو expected = None (يعني مش معروف)، نعرض بس النتيجة
                if expected is None:
                    print(f"{func_name}{inp} -> {output}")
                else:
                    print(f"{func_name}{inp} -> {output} (expected {expected})")
                    if output == expected:
                        correct += 1
            except Exception as e:
                print(f"⚠️ Error calling {func_name}: {e}")
        
        acc = correct / len([t for t in test_cases if t[1] is not None])
        print(f"🎯 Accuracy for {func_name}: {acc*100:.1f}%\n")
        best_acc = max(best_acc, acc)
    
    print(f"✅ Final Overall Accuracy: {best_acc*100:.1f}%")
    return best_acc


In [13]:
code = result["final_answer"]
auto_evaluate_generated_code(code)


❌ Error executing generated code:


Traceback (most recent call last):
  File "C:\Users\ayaha\AppData\Local\Temp\ipykernel_10680\4123192059.py", line 70, in auto_evaluate_generated_code
    exec(clean_code, namespace)
  File "<string>", line 1
    To print "Hello, World!" in Python, you can use the `print()` function. Here’s a simple example:
       ^
SyntaxError: invalid syntax


0